# Webinar 12: Sprint 3: Grammy Nominees & Winners (1958–2024)
Bienvenidos a esta sesión de análisis exploratorio de datos.

## Objetivos
- Realizar un EDA a un dataset.
- Corregir nombres de las columnas.
- Imputar valores vacios y duplicados.
- Proponer agrupaciones de atributos para realizar análisis.

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/data_python_classes/main/DA_DS/images/3_12.jpg" width="300">
</div>

## 1. Importación de librerías y carga de datos
Primero, importaremos pandas y cargaremos el dataset:
- **¿Por qué pandas?** Porque es la biblioteca estándar de Python para manipulación de datos.

In [ ]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv('/mnt/data/Grammy_Nominees_Winners_1958to2024.csv')
df.head()

**Pregunta al aire:** ¿Qué observas en la estructura de las primeras filas?

## 2. EDA básico (sin gráficas)
Veremos la información general y los valores faltantes.
### 2.1 info()
- Muestra columnas, tipos y conteo de valores no nulos.
- **Ejercicio:** Identifica cuántas columnas son numéricas y cuántas categóricas.

In [ ]:
df.info()

### 2.2 describe()
- Estadísticos descriptivos.
- **Pregunta:** ¿Cuál es el rango intercuartílico (IQR) de la duración de canciones nominadas?

In [ ]:
df.describe(include='all')

### 2.3 isnull().sum()
- Cuenta nulos por columna.
- **Ejercicio:** Anota qué columnas tienen más del 10% de valores faltantes.

In [ ]:
null_pct = df.isnull().sum() / len(df) * 100
null_pct[null_pct > 10]

## 3. Filtros básicos
### 3.1 Filtrar ganadores
- `df[df['Winner'] == True]`
- **Pregunta:** ¿Qué artista tiene más premios Grammy en este dataset?

In [ ]:
winners = df[df['Winner'] == True]
winners['Artist'].value_counts().head()

### 3.2 Filtrar por categoría
- Ejemplo: Record of the Year
- **Ejercicio:** Filtra la categoría 'Album of the Year' y cuenta el número de nominaciones.

In [ ]:
album_nom = df[df['Category'] == 'Album of the Year']
album_nom.shape[0]

### 3.3 Filtrar por año
- Ejercicio interactivo: ¿Cuántas nominaciones hubo en 2020?

In [ ]:
df[df['Year'] == 2020].shape[0]

## 4. Cambios de nombres de columna
Estandarizamos nombres para facilitar el análisis.
### 4.1 Usando un bucle for
- Convertir a minúsculas y snake_case.
- **¿Por qué estandarizar?** Evita errores de referencia.

In [ ]:
for col in df.columns:
    new = col.strip().lower().replace(' ', '_').replace('/', '_')
    df.rename(columns={col: new}, inplace=True)
df.columns

### 4.2 Verificación rápida
- Ejercicio: Muestra las columnas que ahora contienen 'year' o 'winner'.

In [ ]:
[col for col in df.columns if 'year' in col or 'winner' in col]

## 5. Búsqueda e imputación de vacíos y nulos
### 5.1 Identificación de nulos
- `df.isnull().sum()`
- **Ejercicio:** Lista las 3 columnas con más nulos.

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(3)

### 5.2 Imputación con forward-fill y backward-fill
- Ejercicio: ¿Qué sucede si aplicas `ffill` y luego `bfill`? Comprueba los cambios.

In [ ]:
df_ffill = df.fillna(method='ffill').fillna(method='bfill')
df_ffill.isnull().sum()

### 5.3 Imputación con valor fijo
- Para columnas categóricas como 'nominee', usaremos 'Unknown'.
- **Ejercicio:** Imputa 'Unknown' en 'nominee'.

In [ ]:
df_ffill['nominee'].fillna('Unknown', inplace=True)
df_ffill['nominee'].isnull().sum()

## 6. `groupby` con funciones
Encapsulamos lógicas comunes en funciones reutilizables.

### 6.1 Conteo de ganadores por año
- Define `count_winners_by_year(df)` que devuelva un Series.

In [ ]:
def count_winners_by_year(df):
    return df[df['winner'] == True].groupby('year')['winner'].count()

count_winners_by_year(df_ffill).head()

**Discusión:** ¿Por qué agrupar antes de filtrar podría cambiar el resultado?

### 6.2 Promedio de nominaciones por categoría
- Define `avg_nominees_per_category(df)` y explica su lógica.

In [ ]:
def avg_nominees_per_category(df):
    counts = df.groupby('category')['nominee'].count()
    return counts.mean()

avg_nominees_per_category(df_ffill)

## Reto final
1. **Pregunta abierta:** ¿Qué otras columnas crees que podríamos analizar con groupby? Propón dos ejemplos.
2. **Desafío práctico:** Combina filtros y groupby para encontrar el artista con mayor número de nominaciones en los últimos 5 años.